In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/GROWTH_VELOCITY.csv', low_memory=False)

In [4]:
column_mapping = {
    'PTNT_ID': 'patient_id',
    'WHO_ENC_ID': 'measure_id',
    'PAT_MRN_ID': 'patient_medical_record',
    'SEX': 'sex',
    'RACE': 'race',
    'ETHNICITY': 'ethnicity',
    'PT_LANGUAGE': 'language',
    'ADI_NATRANK': 'national_adi',
    'ADI_STATERNK': 'state_adi',
    'AGE_AT_ENCOUNTER': 'age_at_encounter',
    'WEIGHT_MEASURED': 'weight_measured',
    'LENGTH_MEASURED': 'length_measured',
    'LENGTH_ZSCORE': 'LAZ',
    'WEIGHT_ZSCORE': 'WAZ',
    'LENGTH_WEIGHT_ZSCORE': 'WLZ',
    'CL_REPAIR_STAY_DAYS': 'CL_repair_stay_days',
    'CL_REPAIR_AGE': 'CL_repair_age',
    'CLEFTLIPREPAIR': 'CL_repair',
    'CP_REPAIR_STAY_DAYS': 'CP_repair_stay_days',
    'CP_REPAIR_AGE': 'CP_repair_age',
    'CLEFTPALATEREPAIR': 'CP_repair',
    'LENGTH_OF_STAY_DAYS': 'length_of_stay_days',
    'LENGTH_OF_STAY_AGE': 'length_of_stay_age',
    'HOSP_COUNT': 'HOSP_count',
    'PHENOTYPEGROUP': 'cleft_type',
    'SYNDROME': 'syndrome'
}

# Create subset with renamed columns
df_subset = df[list(column_mapping.keys())].rename(columns=column_mapping)


In [5]:
# Step 1
data_cleaned = df_subset.dropna(subset=['WAZ']).copy()
print("After dropna:", data_cleaned['patient_id'].nunique(), "patients,", data_cleaned.shape[0], "rows")

# Step 2
data_cleaned['age_at_encounter_months'] = data_cleaned['age_at_encounter'] / 30.4

# Step 3 - match R: if ANY value in group is NA, range = NA
grp = data_cleaned.groupby(['patient_id', 'age_at_encounter'])
data_cleaned['n_day']        = grp['age_at_encounter'].transform('count')
data_cleaned['range_weight'] = grp['weight_measured'].transform(
    lambda x: np.nan if x.isna().any() else x.max() - x.min()
)
data_cleaned['range_length'] = grp['length_measured'].transform(
    lambda x: np.nan if x.isna().any() else x.max() - x.min()
)

print("NA range_length:", data_cleaned['range_length'].isna().sum())  # should be ~26767
print("NA range_weight:", data_cleaned['range_weight'].isna().sum())
print("Rows with n_day>=2 and NA range_length:",
      ((data_cleaned['n_day'] >= 2) & data_cleaned['range_length'].isna()).sum())
print("Rows with n_day>=2 and range_weight>0.1:",
      ((data_cleaned['n_day'] >= 2) & (data_cleaned['range_weight'] > 0.1)).sum())
print("Rows with n_day>=2 and range_length>0.7:",
      ((data_cleaned['n_day'] >= 2) & (data_cleaned['range_length'] > 0.7)).sum())

# Step 4
data_cleaned = data_cleaned[
    ~(
        (data_cleaned['n_day'] >= 2) &
        (
            (data_cleaned['range_weight'] > 0.1) |
            (data_cleaned['range_length'] > 0.7) |
            (data_cleaned['range_length'].isna())
        )
    )
].copy()
print("After range filter:", data_cleaned['patient_id'].nunique(), "patients,", data_cleaned.shape[0], "rows")

# Step 5
grp2 = data_cleaned.groupby(['patient_id', 'age_at_encounter'])
data_cleaned['weight_measure'] = grp2['weight_measured'].transform('mean')
data_cleaned['WAZ']            = grp2['WAZ'].transform('mean')
print("After averaging:", data_cleaned['patient_id'].nunique(), "patients,", data_cleaned.shape[0], "rows")

# Step 6
data_cleaned = data_cleaned.drop_duplicates(subset=['patient_id', 'age_at_encounter']).reset_index(drop=True)
print("After dedup:", data_cleaned['patient_id'].nunique(), "patients,", data_cleaned.shape[0], "rows")

After dropna: 1360 patients, 38286 rows
NA range_length: 33198
NA range_weight: 0
Rows with n_day>=2 and NA range_length: 26767
Rows with n_day>=2 and range_weight>0.1: 3558
Rows with n_day>=2 and range_length>0.7: 240
After range filter: 1207 patients, 11152 rows
After averaging: 1207 patients, 11152 rows
After dedup: 1207 patients, 10050 rows


In [10]:
# --- 2. COHORT SELECTION ---

data_cohort = (
    data_cleaned
    # 1. Primary Filters (Diagnosis, Age, and Sex)
    #.query("cleft_type == 'CLEFT PALATE'")
    #.query("sex == 'M'")
    .query("age_at_encounter < 548")

    # 2. Add Time Calculations
    .assign(age_at_encounter_months=lambda df: df['age_at_encounter'] / 30.4)
)

# 3. Grouped Logic (Patient-level constraints)
grp = data_cohort.groupby('patient_id')
data_cohort = data_cohort.assign(
    n_measures=grp['age_at_encounter'].transform('count'),
    first_visit=grp['age_at_encounter'].transform('min')
)

# 4. Longitudinal Criteria
data_cohort = (
    data_cohort
    .query("n_measures >= 4 & n_measures <= 10")
    .query("first_visit < 30")
)

# filter(any(age_at_encounter > 200)) — keep patients who have AT LEAST ONE visit > 200 days
patients_with_late_visit = (
    data_cohort
    .groupby('patient_id')['age_at_encounter']
    .apply(lambda x: (x > 200).any())
)
valid_patients = patients_with_late_visit[patients_with_late_visit].index
data_cohort = data_cohort[data_cohort['patient_id'].isin(valid_patients)].reset_index(drop=True)

# Number of unique patients in cohort after filters
print("Unique patients in cohort:", data_cohort['patient_id'].nunique())

Unique patients in cohort: 87


In [7]:
data_cohort.groupby('cleft_type').agg(
    n_patients=('patient_id', 'nunique'),
    n_rows=('patient_id', 'count')
).reset_index()

,cleft_type,n_patients,n_rows
0,CLEFT LIP,6,32
1,CLEFT LIP AND PALATE,48,294
2,CLEFT PALATE,33,215


In [11]:
attrition_tracking = {}

# 0. Starting Point
attrition_tracking['Start'] = data_cleaned['patient_id'].nunique()

# 1. Primary Filters (Diagnosis, Age, and Sex)
step1 = data_cleaned[
    #(data_cleaned['cleft_type'] == 'CLEFT PALATE') &
    #(data_cleaned['sex'] == 'M') &
    (data_cleaned['age_at_encounter'] < 548)
].copy()
attrition_tracking['Diagnosis_Sex_Age'] = step1['patient_id'].nunique()

# 2. Longitudinal: Measure Count (3 to 6 visits)
step2 = step1.copy()
step2['n_measures'] = step2.groupby('patient_id')['age_at_encounter'].transform('count')
step2 = step2[step2['n_measures'].between(4, 10)].copy()
attrition_tracking['Measure_Count_3_to_6'] = step2['patient_id'].nunique()

# 3. Longitudinal: Follow-up Duration (Any visit > 200 days)
has_late_visit = (
    step2.groupby('patient_id')['age_at_encounter']
    .apply(lambda x: (x > 200).any())
)
valid_patients = has_late_visit[has_late_visit].index
step3 = step2[step2['patient_id'].isin(valid_patients)].copy()
attrition_tracking['FollowUp_After_Day200'] = step3['patient_id'].nunique()

# 4. Longitudinal: Early Entry (First visit < 30 days)
step3['first_visit'] = step3.groupby('patient_id')['age_at_encounter'].transform('min')
data_cohort = step3[step3['first_visit'] < 30].reset_index(drop=True)
attrition_tracking['Entry_Before_Day30'] = data_cohort['patient_id'].nunique()

# 5. CREATE ATTRITION TABLE
attrition_table = pd.DataFrame({
    'Step': list(attrition_tracking.keys()),
    'Remaining_Patients': list(attrition_tracking.values())
})

attrition_table['Dropped'] = attrition_table['Remaining_Patients'].shift(1) - attrition_table['Remaining_Patients']
attrition_table['Percent_Retained'] = (
    (attrition_table['Remaining_Patients'] / attrition_table['Remaining_Patients'].iloc[0]) * 100
).round(1)

attrition_table

,Step,Remaining_Patients,Dropped,Percent_Retained
0,Start,1207,NaN,100.0
1,Diagnosis_Sex_Age,987,220.0,81.8
2,Measure_Count_3_to_6,271,716.0,22.5
3,FollowUp_After_Day200,239,32.0,19.8
4,Entry_Before_Day30,87,152.0,7.2
